# 4.1 Векторы и матрицы в NumPy: код страницы и упражнения

Аналитика 101 · блок «Дата аналитика» · тема 04 «Математика для данных» · страница 4.1.

В ноутбуке шесть примеров из урока и пять упражнений на форму массивов, оси и broadcasting. Данные о тысяче клиентов находятся в наборе темы 04 (`data/clients/`), а данные о покупках — в базе магазина из темы 03 (`data/shop/`).

**Как работать.** Выполняйте ячейки по порядку сверху вниз. В упражнении замените `...` своим кодом и запустите ячейку проверки под ним. Она напечатает «верно» или покажет, что не сошлось. Решения с разбором — в `vectors_numpy_solutions.ipynb`.

Вывод под блоками кода получен на Python 3.13, NumPy 2.5 и pandas 3.0.

## Где лежат данные

Ноутбук читает файлы `data/clients/clients.csv` и `data/shop/shop.sqlite`. Первая ячейка ищет папку `data` в текущем каталоге и выше по дереву каталогов. Папку, в которой находится `data`, она назначает рабочим каталогом.

In [1]:
import os
import pathlib
import platform

import numpy as np
import pandas as pd

root = pathlib.Path.cwd()
while not (root / "data" / "clients").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/clients/
assert (root / "data" / "clients" / "clients.csv").exists(), \
    "Не найдена папка data/clients/. Скачайте общий набор темы 04 и положите data/ рядом с папкой страницы"
assert (root / "data" / "shop" / "shop.sqlite").exists(), \
    "Не найдена папка data/shop/. Скачайте общий набор темы 03 и положите его в ту же папку data/"
os.chdir(root)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

Python 3.13.13 | numpy 2.5.0 | pandas 3.0.5


## Код страницы

Блоки идут в том же порядке, что на странице: сначала три блока из раздела «Нюансы», затем три из раздела «Код». Каждый блок заново читает таблицу, поэтому их можно запускать по отдельности.

### Одномерный массив (5,) и столбец (5, 1)

In [2]:
import numpy as np
import pandas as pd
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()

age = X[:5, 0]                 # один индекс — ось пропадает
print(age, age.shape)
print(X[:5, [0]].shape, X[:5, 0:1].shape)   # список или срез — ось остаётся
col = age.reshape(-1, 1)       # то же, что age[:, None]
print(col.shape)
print(age.T.shape)             # транспонирование сохраняет форму одномерного массива
print(col.T.shape)

[66 34 48 41 59] (5,)
(5, 1) (5, 1)
(5, 1)
(5,)
(1, 5)


После транспонирования одномерный массив сохраняет форму (5,). Чтобы получить из него столбец с пятью строками, используйте reshape(-1, 1) или добавьте ось с помощью [:, None].

### Почему получается миллион разностей вместо тысячи

In [3]:
import numpy as np
import pandas as pd
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()

a = np.array([1, 2, 3])           # (3,)
b = np.array([[10], [20], [30]])  # (3, 1)
print(a + b)                      # элемент каждого вектора складывается с каждым элементом другого
print((a + b).shape)

X1 = np.c_[np.ones(len(X)), X[:, 1:]]
w = np.array([205, 0.0347, 2550])
spend = df["spend"].to_numpy()          # (1000,)
pred = (X1 @ w).reshape(-1, 1)          # (1000, 1): прогноз случайно стал столбцом
err = spend - pred                      # (1000,) − (1000, 1) → (1000, 1000), без ошибки
print(err.shape)
print(np.abs(err).mean().round())               # среднее абсолютное значение разности по всем парам
print(np.abs(spend - pred.ravel()).mean().round())  # средняя абсолютная ошибка по клиентам

[[11 12 13]
 [21 22 23]
 [31 32 33]]
(3, 3)
(1000, 1000)
7532.0
3017.0


Формы совместимы по правилам broadcasting в обоих случаях. При сопоставлении каждой фактической суммы со всеми прогнозами среднее абсолютное значение разностей равно 7 532 ₽. Средняя абсолютная ошибка по парам «факт и прогноз одного клиента» равна 3 017 ₽.

### Что происходит при записи дробного числа в целочисленный массив и при переполнении

In [4]:
import numpy as np
import pandas as pd
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()
print(X.dtype)                     # все столбцы целые → int64

X[0, 1] = 80_500.7                 # в целочисленном массиве сохранится только целая часть
print(X[0])

try:
    X -= X.mean(axis=0)            # результат float64 нельзя записать обратно по правилам этой операции
except TypeError as e:
    print(type(e).__name__, e)
Xc = X - X.mean(axis=0)            # новый массив: NumPy сам повышает до float64
print(Xc.dtype)

inc32 = X[:3, 1].astype(np.int32)  # переводим доходы в 32-битные целые
print(inc32 * inc32)               # 80 500² > 2 147 483 647: переполнение без предупреждения
print(X[:3, 1] * X[:3, 1])         # в int64 верно
print(X[:3, 1] // 1000, X[:3, 1] / 1000)   # // округляет частное вниз, / даёт вещественное частное

int64
[   66 80500     2]
UFuncTypeError Cannot cast ufunc 'subtract' output from dtype('float64') to dtype('int64') with casting rule 'same_kind'
float64
[-2109684592   625000000   466032704]
[6480250000  625000000 4761000000]
[80 25 69] [80.5 25.  69. ]


UFuncTypeError — подкласс TypeError. Наибольшее значение типа int32 равно 2 147 483 647. Квадрат числа 80 500 равен 6 480 250 000, поэтому при умножении массивов типа int32 произошло переполнение.

### Форма матрицы клиентов и прогноз квартальных трат

In [5]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=3, suppress=True)

a = np.array([57, 118_000, 6])          # клиент 49: возраст, доход, заказы
print(a, a.shape, a.ndim, a.dtype)

df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()   # матрица «клиенты × признаки»
print(X.shape, X.dtype)

print(X[48])              # строка 48 — клиент 49 (счёт с нуля)
print(X[:3, 1])           # срез: доход первых трёх клиентов
print(X[:, 2].max())      # столбец «заказы» целиком, максимум
print(X[X[:, 2] >= 12])   # строки по условию: 12 заказов и больше

X1 = np.c_[np.ones(len(X)), X[:, 1:]]   # (1000, 3): единица, доход, заказы
w = np.array([205, 0.0347, 2550])        # коэффициенты прогноза трат (подбор весов — в уроке 4.2)
pred = X1 @ w                            # скалярное произведение каждой строки и w
print(X1.shape, w.shape, pred.shape)
print(pred[:3], df["spend"].to_numpy()[:3])

[    57 118000      6] (3,) 1 int64
(1000, 3) int64
[    57 118000      6]
[80000 25000 69000]
15
[[    60 314000     15]
 [    44 273000     12]
 [    70 185000     12]]
(1000, 3) (3,) (1000,)
[ 8081.   8722.5 20449.3] [ 9080 10020 15900]


В X1 три столбца: единица, месячный доход и число заказов. Выражение X1 @ w даёт по одному прогнозу для каждого клиента. Сравните первые три прогноза с фактическими тратами, напечатанными рядом.

### Попарные расстояния между клиентами

In [6]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=2, suppress=True)
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()

Z = (X - X.mean(axis=0)) / X.std(axis=0)   # стандартизация: (1000, 3) − (3,), потом / (3,)

S = Z[:6]                                  # первые 6 клиентов
diff = S[:, None, :] - S[None, :, :]       # (6, 1, 3) − (1, 6, 3) → (6, 6, 3)
D = np.sqrt((diff ** 2).sum(axis=-1))      # сумма по признакам → (6, 6)
print(diff.shape, D.shape)
print(D)

R = X[:6]                                  # те же клиенты в исходных единицах
print(np.sqrt(((R[:, None, :] - R[None, :, :]) ** 2).sum(axis=-1)).round())

# все 1000 × 1000 через скалярные произведения: |a − b|² = |a|² + |b|² − 2 a·b
sq = (Z ** 2).sum(axis=1)                            # (1000,)
D2 = sq[:, None] + sq[None, :] - 2 * Z @ Z.T         # (1000, 1000)
D_all = np.sqrt(np.clip(D2, 0, None))                # отрицательные значения около нуля возникают из-за округления
near = np.argsort(D_all[48])[:4]                     # клиент 49 и трое ближайших
print(df.iloc[near, :4].assign(dist=D_all[48, near].round(3)).to_string(index=False))

(6, 6, 3) (6, 6)
[[0.   2.56 2.59 1.71 1.04 2.46]
 [2.56 0.   2.35 1.5  2.09 0.18]
 [2.59 2.35 0.   2.8  1.56 2.27]
 [1.71 1.5  2.8  0.   1.81 1.37]
 [1.04 2.09 1.56 1.81 0.   1.99]
 [2.46 0.18 2.27 1.37 1.99 0.  ]]
[[    0. 55000. 11000. 13000.  7000. 48000.]
 [55000.     0. 44000. 42000. 48000.  7000.]
 [11000. 44000.     0.  2000.  4000. 37000.]
 [13000. 42000.  2000.     0.  6000. 35000.]
 [ 7000. 48000.  4000.  6000.     0. 41000.]
 [48000.  7000. 37000. 35000. 41000.     0.]]
 client_id  age  income  orders  dist
        49   57  118000       6 0.000
       960   59  122000       6 0.166
       745   59  109000       6 0.268
       387   62  112000       6 0.358


Для первых шести клиентов показаны две матрицы расстояний. В исходных единицах (вторая матрица) ближе всего клиенты 3 и 4 с доходами 69 000 и 67 000 ₽. После стандартизации (первая матрица) между ними наибольшее расстояние среди этих шести клиентов — 2,80. Один сделал семь заказов, другой — один. При расчёте по скалярным произведениям возможна погрешность округления. clip заменяет малые отрицательные значения квадратов расстояний нулём перед извлечением корня.

### По составу покупателей к пледу ближе всего три товара того же бренда

In [7]:
import sqlite3
import numpy as np
import pandas as pd
np.set_printoptions(precision=3, suppress=True)

con = sqlite3.connect("data/shop/shop.sqlite")
long = pd.read_sql("""
    SELECT o.customer_id, p.product_name, SUM(oi.qty) AS qty
    FROM order_items AS oi
    JOIN orders   AS o USING (order_id)
    JOIN products AS p USING (product_id)
    WHERE o.status = 'paid'
    GROUP BY o.customer_id, p.product_name""", con)

M = long.pivot_table(index="customer_id", columns="product_name",
                     values="qty", fill_value=0)       # клиенты × товары
P = M.to_numpy(dtype=float)
print(P.shape)

a = P[:, M.columns.get_loc("Плед «Сова»")]             # столбец: кто сколько купил
b = P[:, M.columns.get_loc("Наволочка «Сова»")]
print(a @ b, np.linalg.norm(a).round(3), np.linalg.norm(b))
print(round(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)), 3))

Pn = P / np.linalg.norm(P, axis=0)                     # (51, 32) / (32,): столбцы длины 1
C = Pn.T @ Pn                                          # (32, 32): косинусы всех пар
i = M.columns.get_loc("Плед «Сова»")
top = np.argsort(-C[i])[1:4]                           # [0] — сам плед, cos = 1
print(pd.Series(C[i, top], index=M.columns[top]).round(3).to_string())
print((C[i] == 0).sum(), "товаров без единого общего покупателя: cos = 0")

(51, 32)
44.0 9.165 8.0
0.6
product_name
Свеча «Сова»        0.767
Чайник «Сова»       0.706
Наволочка «Сова»    0.600
10 товаров без единого общего покупателя: cos = 0


Из 60 клиентов оплатили хотя бы один заказ 51, поэтому в матрице 51 строка. Для пледа и наволочки a @ b = 44. Клиент, который купил оба товара, вносит в эту сумму произведение двух количеств. Три товара с наибольшим косинусом для пледа принадлежат «Сове». У десяти других товаров с пледом нет общих покупателей, и косинус равен нулю.

## Упражнения

Запишем признаки в массив `X` с вещественными элементами и стандартизуем каждый столбец. В матрице `Z` координаты показывают отклонения от среднего в единицах стандартного отклонения.

In [8]:
np.set_printoptions(precision=3, suppress=True)
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy(dtype=float)   # возраст, доход, заказы
Z = (X - X.mean(axis=0)) / X.std(axis=0)                    # те же клиенты в стандартных единицах
print(X.shape, X.dtype, Z.shape)
print(Z[48])                                                # клиент 49

(1000, 3) float64 (1000, 3)
[0.851 1.012 0.932]


### Упражнение 1. Средние по столбцам и максимумы по строкам

Запишите в `col_mean` средние трёх признаков по всем клиентам, а в `row_max` — наибольшее из трёх чисел в строке каждого клиента. Используйте массив `X` с исходными значениями.

In [ ]:
col_mean = ...   # ваш код: по числу на признак
row_max = ...    # ваш код: по числу на клиента

In [ ]:
assert getattr(col_mean, "shape", None) == (3,), f"col_mean — по числу на признак, форма (3,); получено {getattr(col_mean, 'shape', None)}. Строки сворачивает axis=0"
assert np.allclose(col_mean, [43.802, 79323.0, 3.975]), f"Средние: возраст 43,802, доход 79 323, заказы 3,975; получено {col_mean}"
assert getattr(row_max, "shape", None) == (1000,), f"row_max — по числу на клиента, форма (1000,); получено {getattr(row_max, 'shape', None)}. Столбцы сворачивает axis=1"
assert np.allclose(row_max, X[:, 1]), "В каждой строке самое большое число — доход: row_max должен совпасть со столбцом X[:, 1]"
print("Упражнение 1: верно")

### Упражнение 2. Строки длины 1

Разделите каждый вектор-строку матрицы `Z` на его длину. Запишите результаты в матрицу `U`. После этой операции каждая строка должна иметь единичную длину и сохранять исходное направление. В учебных данных все строки `Z` ненулевые.

In [ ]:
U = ...   # ваш код: (1000, 3) делим на длины строк

In [ ]:
assert getattr(U, "shape", None) == (1000, 3), f"U должна остаться матрицей (1000, 3); получено {getattr(U, 'shape', None)}"
assert np.allclose(np.linalg.norm(U, axis=1), 1), "Длина каждой строки U должна быть 1: разделите строку на её длину, а не столбец на его длину"
assert np.allclose(U[48], [0.526, 0.626, 0.576], atol=1e-3), f"Строка клиента 49 должна быть (0,526; 0,626; 0,576); получено {U[48]}"
print("Упражнение 2: верно")

### Упражнение 3. Расстояния от одного клиента до всех

Запишите в `d` расстояния от клиента 49 до каждого из 1 000 клиентов в стандартных единицах — без цикла. В `nearest` запишите идентификаторы `client_id` трёх ближайших к нему клиентов, не считая его самого.

In [ ]:
d = ...         # ваш код: форма (1000,)
nearest = ...   # ваш код: три номера client_id

In [ ]:
assert getattr(d, "shape", None) == (1000,), f"d — по расстоянию на клиента, форма (1000,); получено {getattr(d, 'shape', None)}"
assert d[48] == 0, f"Расстояние от клиента 49 до самого себя — 0; получено {d[48]}"
assert np.allclose(d[[959, 744, 386]], [0.166, 0.268, 0.358], atol=1e-3), "До клиентов 960, 745 и 387 должно быть 0,166, 0,268 и 0,358. Считайте по Z, а не по X"
assert list(nearest) == [960, 745, 387], f"Ближайшие — клиенты 960, 745 и 387; получено {list(nearest)}. Строка 48 — это client_id 49"
print("Упражнение 3: верно")

### Упражнение 4. Косинус угла между вектором клиента и каждой строкой матрицы

Для каждого клиента вычислите косинус угла между его вектором и вектором клиента 49 после стандартизации. Запишите результаты в `cos`. В `same_side` сохраните число строго положительных косинусов. Включите в расчёт сравнение клиента 49 с самим собой.

In [ ]:
cos = ...         # ваш код: форма (1000,)
same_side = ...   # ваш код: целое число

In [ ]:
assert getattr(cos, "shape", None) == (1000,), f"cos — по числу на клиента, форма (1000,); получено {getattr(cos, 'shape', None)}"
assert np.isclose(cos[48], 1), f"Косинус клиента 49 с самим собой — 1; получено {cos[48]}"
assert np.isclose(cos[880], -0.928, atol=1e-3), f"С клиентом 881 косинус −0,928 (угол 158°); получено {cos[880]}. Поделили на обе длины?"
assert same_side == 461, f"Положительный косинус получился для 461 клиента; получено {same_side}"
print("Упражнение 4: верно")

### Упражнение 5. Форма массива и ошибка прогноза

Прогнозы `pred` хранятся в столбце формы (1000, 1), а фактические траты `spend` — в массиве формы (1000,). Запишите в `bad_shape` форму результата `spend - pred`. Затем сравните прогноз каждого клиента с его фактическими тратами и сохраните среднюю абсолютную ошибку в `mae`.

In [ ]:
X1 = np.c_[np.ones(len(df)), df["income"], df["orders"]]
w = np.array([205, 0.0347, 2550])
spend = df["spend"].to_numpy()             # (1000,)
pred = (X1 @ w).reshape(-1, 1)             # (1000, 1)
print(spend.shape, pred.shape)

In [ ]:
bad_shape = ...   # ваш код: форма spend - pred
mae = ...         # ваш код: средняя абсолютная ошибка, ₽

In [ ]:
assert bad_shape == (1000, 1000), f"(1000,) минус (1000, 1) даёт таблицу (1000, 1000); получено {bad_shape}"
assert isinstance(mae, (int, float, np.floating)) and round(float(mae)) == 3017, f"Средняя абсолютная ошибка по клиентам — 3 017 ₽; получено {mae}. 7 532 ₽ получится, если сопоставить траты каждого клиента со всеми прогнозами"
print("Упражнение 5: верно")